In [1]:
%%capture
!pip install -q "transformers==4.51.3" "trl==0.18.2" "peft==0.14.0" "bitsandbytes>=0.49.0" "accelerate>=1.0.0" datasets

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---- Base model ----
MODEL_NAME      = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH  = 1024

# ---- LoRA ----
LORA_R          = 32
LORA_ALPHA      = 32
LORA_DROPOUT    = 0

# ---- Training ----
BATCH_SIZE      = 1
GRAD_ACCUM      = 16
EPOCHS          = 1
LEARNING_RATE   = 1e-4
MAX_STEPS       = 500

# ---- Paths ----
DATASET_DIR     = "/kaggle/input/datasets/maximuz23/osint-ai-dataset"
TRAIN_FILE      = f"{DATASET_DIR}/train.jsonl"
VALID_FILE      = f"{DATASET_DIR}/valid.jsonl"
TEST_FILE       = f"{DATASET_DIR}/test.jsonl"
OUTPUT_DIR      = "/kaggle/working/osint-ai"
HF_REPO         = "Maximuz23/Text-OSINT"

# ---- HuggingFace token ----
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map={"": 0},
    token=HF_TOKEN,
    torch_dtype=torch.float16,
)
model.config.use_cache = False

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

2026-05-18 15:10:12.679959: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779117012.911910      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779117012.975166      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779117013.507297      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779117013.507324      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779117013.507328      23 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

In [4]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    task_type      = "CAUSAL_LM",
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} out of {total:,} ({100*trainable/total:.2f}%)")

Trainable: 48,627,712 out of 1,852,091,392 (2.63%)


In [5]:
from datasets import load_dataset

raw = load_dataset("json", data_files={
    "train": TRAIN_FILE,
    "valid": VALID_FILE,
    "test":  TEST_FILE,
})

def format_messages(examples):
    return {"text": [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in examples["messages"]
    ]}

train_dataset = raw["train"].map(format_messages, batched=True, remove_columns=raw["train"].column_names)
valid_dataset = raw["valid"].map(format_messages, batched=True, remove_columns=raw["valid"].column_names)
test_dataset  = raw["test" ].map(format_messages, batched=True, remove_columns=raw["test" ].column_names)

print(f"Train set: {len(train_dataset):,}")
print(f"Valid set: {len(valid_dataset):,}")
print(f"Test set:  {len(test_dataset):,}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating valid split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/36552 [00:00<?, ? examples/s]

Map:   0%|          | 0/3070 [00:00<?, ? examples/s]

Map:   0%|          | 0/3082 [00:00<?, ? examples/s]

Train set: 36,552
Valid set: 3,070
Test set:  3,082


In [6]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

EVAL_STEPS = 100

sft_config = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    num_train_epochs            = EPOCHS,
    max_steps                   = MAX_STEPS,
    learning_rate               = LEARNING_RATE,
    warmup_ratio                = 0.05,
    lr_scheduler_type           = "cosine",
    weight_decay                = 0.01,
    fp16                        = True,
    bf16                        = False,
    optim                       = "paged_adamw_8bit",
    seed                        = 42,
    logging_steps               = 25,
    eval_strategy               = "steps",
    eval_steps                  = EVAL_STEPS,
    save_strategy               = "steps",
    save_steps                  = EVAL_STEPS,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    report_to                   = "none",
    dataloader_num_workers      = 2,
    gradient_checkpointing      = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},
    dataset_text_field          = "text",
    max_length                  = MAX_SEQ_LENGTH,
    packing                     = True,
    dataset_num_proc            = 1,
)

trainer = SFTTrainer(
    model            = model,
    args             = sft_config,
    train_dataset    = train_dataset,
    eval_dataset     = valid_dataset,
    processing_class = tokenizer,
    callbacks        = [EarlyStoppingCallback(
        early_stopping_patience  = 2,
        early_stopping_threshold = 0.001,
    )],
)

Converting train dataset to ChatML (num_proc=1):   0%|          | 0/36552 [00:00<?, ? examples/s]

Adding EOS to train dataset (num_proc=1):   0%|          | 0/36552 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=1):   0%|          | 0/36552 [00:00<?, ? examples/s]

Packing train dataset (num_proc=1):   0%|          | 0/36552 [00:00<?, ? examples/s]

Converting eval dataset to ChatML (num_proc=1):   0%|          | 0/3070 [00:00<?, ? examples/s]

Adding EOS to eval dataset (num_proc=1):   0%|          | 0/3070 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=1):   0%|          | 0/3070 [00:00<?, ? examples/s]

Packing eval dataset (num_proc=1):   0%|          | 0/3070 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [7]:
stats = trainer.train()

print()
print(f"Train runtime {stats.metrics['train_runtime']/3600:.2f} hours.")
print(f"Train loss: {stats.metrics['train_loss']:.4f}")

Step,Training Loss,Validation Loss
100,0.894000,0.838702
200,0.795600,0.794392
300,0.766100,0.778397
400,0.760400,0.770440
500,0.730700,0.768823



Train runtime 5.88 hours.
Train loss: 0.9136


In [8]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Maximuz23/Text-OSINT/commit/3ba4ae67bb502a73e82097f005d8ecf45cb5d0e5', commit_message='Upload tokenizer', commit_description='', oid='3ba4ae67bb502a73e82097f005d8ecf45cb5d0e5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Maximuz23/Text-OSINT', endpoint='https://huggingface.co', repo_type='model', repo_id='Maximuz23/Text-OSINT'), pr_revision=None, pr_num=None)

In [9]:
def tokenize_for_eval(batch):
    enc = tokenizer(batch["text"], truncation=True, max_length=MAX_SEQ_LENGTH)
    enc["labels"] = [ids.copy() for ids in enc["input_ids"]]
    return enc

test_tokenized = test_dataset.map(tokenize_for_eval, batched=True, remove_columns=["text"])

test_metrics = trainer.evaluate(eval_dataset=test_tokenized)
print()
print(f"Test loss: {test_metrics['eval_loss']:.4f}")

Map:   0%|          | 0/3082 [00:00<?, ? examples/s]


Test loss: 0.6274
